In [1]:
# ------------------------------
# Import libraries
# ------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

# ------------------------------
# Load dataset
# ------------------------------
file_path = r"../Rice_Cammeo_Osmancik.xlsx"
df = pd.read_excel(file_path)

features = ['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length',
            'Eccentricity', 'Convex_Area', 'Extent']
X = df[features]
y = df['Class']

# ------------------------------
# Helper function
# ------------------------------
def safe_div(a, b):
    return a / b if b != 0 else np.nan

# ------------------------------
# Metrics from confusion matrix
# ------------------------------
def compute_metrics(cm):
    tn, fp, fn, tp = cm.ravel()
    return np.array([
        safe_div(tp + tn, tp + tn + fp + fn),   # Accuracy
        safe_div(tp, tp + fn),                  # Sensitivity
        safe_div(tn, tn + fp),                  # Specificity
        safe_div(tp, tp + fp),                  # Precision
        safe_div(2 * tp, 2 * tp + fp + fn),     # F1
        safe_div(tn, tn + fn),                  # NPV
        safe_div(fp, tn + fp),                  # FPR
        safe_div(fp, tp + fp),                  # FDR
        safe_div(fn, tp + fn)                   # FNR
    ])

# ------------------------------
# Main function (Models + Scalers)
# ------------------------------
def train_models_all_scalers(X, y, cv_splits=10, random_state=42):

    models = {
        "LR": LogisticRegression(max_iter=1000),
        "MLP": MLPClassifier(max_iter=1000),
        "SVM": SVC(),
        "DT": DecisionTreeClassifier(),
        "RF": RandomForestClassifier(),
        "NB": GaussianNB(),
        "k-NN": KNeighborsClassifier(n_neighbors=1)
    }

    scalers = {
        "Unscaled": None,
        "Standard": StandardScaler(),
        "MinMax": MinMaxScaler(),
        "Robust": RobustScaler(),
        "MaxAbs": MaxAbsScaler()
    }

    measures = [
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-Score",
        "Negative Predictive Value",
        "False Positive Rate",
        "False Discovery Rate",
        "False Negative Rate"
    ]

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    results = {}

    for scaler_name, scaler in scalers.items():
        for model_name, model in models.items():

            fold_metrics = []

            for train_idx, test_idx in cv.split(X, y):

                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                # Pipeline with scaler
                if scaler is not None:
                    pipeline = Pipeline([
                        ("scaler", scaler),
                        ("model", model)
                    ])
                else:
                    pipeline = model

                pipeline.fit(X_train, y_train)
                preds = pipeline.predict(X_test)

                cm = confusion_matrix(y_test, preds)
                fold_metrics.append(compute_metrics(cm))

            results[f"{model_name}-{scaler_name}"] = np.nanmean(fold_metrics, axis=0)

    results_df = pd.DataFrame(results, index=measures) * 100
    return results_df

# ------------------------------
# Best model per measure
# ------------------------------
def get_best_models(results_df):

    best_models = results_df.idxmax(axis=1)
    best_values = results_df.max(axis=1)

    print("\nBest model per measure:\n")
    for measure in results_df.index:
        print(f"{measure:30s} -> {best_models[measure]} ({best_values[measure]:.2f}%)")

    print("\nWins per model:\n")
    print(best_models.value_counts())

# ------------------------------
# Run experiment
# ------------------------------
results_df = train_models_all_scalers(X, y)

display(results_df.round(2))

get_best_models(results_df)

,LR-Unscaled,MLP-Unscaled,SVM-Unscaled,DT-Unscaled,RF-Unscaled,NB-Unscaled,k-NN-Unscaled,LR-Standard,MLP-Standard,SVM-Standard,...,RF-Robust,NB-Robust,k-NN-Robust,LR-MaxAbs,MLP-MaxAbs,SVM-MaxAbs,DT-MaxAbs,RF-MaxAbs,NB-MaxAbs,k-NN-MaxAbs
Accuracy,93.04,62.20,88.06,88.11,92.44,91.10,86.67,92.94,92.81,92.89,...,92.34,91.71,88.64,91.78,92.52,92.86,88.35,92.15,91.71,88.50
Sensitivity,94.50,63.30,92.89,89.72,93.62,92.80,88.62,93.99,93.90,94.17,...,93.62,93.26,89.45,94.40,93.67,94.08,90.18,93.44,93.26,89.50
Specificity,91.10,60.74,81.60,85.95,90.86,88.83,84.05,91.53,91.35,91.17,...,90.61,89.63,87.55,88.28,90.98,91.23,85.89,90.43,89.63,87.18
Precision,93.44,79.81,87.13,89.56,93.23,91.77,88.15,93.71,93.58,93.46,...,93.05,92.36,90.62,91.54,93.33,93.50,89.59,92.91,92.36,90.37
F1-Score,93.96,59.78,89.90,89.62,93.41,92.27,88.37,93.84,93.73,93.81,...,93.32,92.78,90.01,92.93,93.47,93.78,89.86,93.16,92.78,89.90
Negative Predictive Value,92.54,63.82,89.61,86.27,91.45,90.26,84.73,91.96,91.83,92.16,...,91.44,90.95,86.16,92.25,91.59,92.05,86.77,91.19,90.95,86.20
False Positive Rate,8.90,39.26,18.40,14.05,9.14,11.17,15.95,8.47,8.65,8.83,...,9.39,10.37,12.45,11.72,9.02,8.77,14.11,9.57,10.37,12.82
False Discovery Rate,6.56,20.19,12.87,10.44,6.77,8.23,11.85,6.29,6.42,6.54,...,6.95,7.64,9.38,8.46,6.67,6.50,10.41,7.09,7.64,9.63
False Negative Rate,5.50,36.70,7.11,10.28,6.38,7.20,11.38,6.01,6.10,5.83,...,6.38,6.74,10.55,5.60,6.33,5.92,9.82,6.56,6.74,10.50



Best model per measure:

Accuracy                       -> LR-Unscaled (93.04%)
Sensitivity                    -> LR-Unscaled (94.50%)
Specificity                    -> LR-Standard (91.53%)
Precision                      -> LR-Standard (93.71%)
F1-Score                       -> LR-Unscaled (93.96%)
Negative Predictive Value      -> LR-Unscaled (92.54%)
False Positive Rate            -> MLP-Unscaled (39.26%)
False Discovery Rate           -> MLP-Unscaled (20.19%)
False Negative Rate            -> MLP-Unscaled (36.70%)

Wins per model:

LR-Unscaled     4
MLP-Unscaled    3
LR-Standard     2
Name: count, dtype: int64
